# Lesson 02 — DFT in OpenCV: Computing and Filtering

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

img  = cv2.imread('sample.jpg')
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY).astype(np.float32)

dft         = cv2.dft(gray, flags=cv2.DFT_COMPLEX_OUTPUT)
dft_shifted = np.fft.fftshift(dft)

h, w = gray.shape
cx, cy = w//2, h//2

# Low-pass filter: keep only center (low frequencies = smooth/blur)
radius = 60
mask_lp = np.zeros((h, w, 2), np.uint8)
cv2.circle(mask_lp, (cx, cy), radius, (1,1), -1)

# High-pass filter: keep only outer (high frequencies = edges)
mask_hp = np.ones((h, w, 2), np.uint8)
cv2.circle(mask_hp, (cx, cy), radius, (0,0), -1)

def apply_freq_filter(dft_shifted, mask):
    filtered     = dft_shifted * mask
    back_shifted = np.fft.ifftshift(filtered)
    result       = cv2.idft(back_shifted, flags=cv2.DFT_SCALE | cv2.DFT_REAL_OUTPUT)
    return cv2.normalize(result, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)

low_pass  = apply_freq_filter(dft_shifted, mask_lp)
high_pass = apply_freq_filter(dft_shifted, mask_hp)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, im, t in zip(axes, [gray.astype(np.uint8), low_pass, high_pass],
    ['Original', 'Low-pass (blur)', 'High-pass (edges)']):
    ax.imshow(im, cmap='gray'); ax.set_title(t); ax.axis('off')
plt.show()

## Key Takeaway
Low-pass filter in frequency domain = Gaussian blur in spatial domain. Same mathematical operation, different domain.